Cellule 1 - Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df = pd.read_csv("../data/produits_clean.csv", parse_dates=["date"])

Cellule 2 - KPI : Score moyen par type chimique

In [ ]:
kpi1 = (
    df.groupby("type_chimique")
    .agg(
        score_moyen=("score_env", "mean"),
        volume_kg=("quantite_kg", "sum"),
        nb_achats=("produit", "count")
    )
    .sort_values("score_moyen", ascending=False)
    .round(2)
)
print("=== Score environnemental par type chimique ===")
kpi1

Cellule 3 - KPI2 : Part éco-responsable en volume

In [ ]:
total_kg  = df["quantite_kg"].sum()
eco_kg    = df[df["est_eco"]]["quantite_kg"].sum()
part_eco  = eco_kg / total_kg * 100

print(f"Volume total acheté    : {total_kg:,.0f} kg")
print(f"Volume éco-responsable : {eco_kg:,.0f} kg")
print(f"Part éco               : {part_eco:.1f} %")

# Graphique
labels  = ["Éco-responsable", "Conventionnel"]
values  = [eco_kg, total_kg - eco_kg]
colors  = ["#4CAF50", "#E57373"]

plt.figure(figsize=(5, 5))
plt.pie(values, labels=labels, colors=colors,
        autopct="%1.1f%%", startangle=90)
plt.title("Part éco-responsable dans les achats (volume)")
plt.tight_layout()
plt.savefig("../reports/04_part_eco.png", dpi=150)
plt.show()

Cellule 4 - KPI 3 : Top 5 produits les plus polluants (charge COV)

In [ ]:
top_cov = (
    df.groupby("produit")
    .agg(
        charge_cov_totale=("charge_cov_g", "sum"),
        cov_moyen=("cov_g_par_l", "mean"),
        volume_kg=("quantite_kg", "sum")
    )
    .sort_values("charge_cov_totale", ascending=False)
    .head(5)
    .round(1)
)

print("=== Top 5 produits — charge COV totale ===")
print(top_cov)

plt.figure(figsize=(10, 4))
plt.barh(top_cov.index, top_cov["charge_cov_totale"], color="#E8724C")
plt.xlabel("Charge COV totale (g)")
plt.title("Top 5 produits les plus émetteurs de COV")
plt.tight_layout()
plt.savefig("../reports/05_top_cov.png", dpi=150)
plt.show()

Cellule 5 - KPI 4 : Evolution annuelle

In [ ]:
tendance = (
    df.groupby("annee")
    .agg(
        score_moyen=("score_env", "mean"),
        part_eco=("est_eco", "mean"),
        charge_cov=("charge_cov_g", "sum")
    )
    .round(3)
)
tendance["part_eco"] = (tendance["part_eco"] * 100).round(1)

print("=== Évolution annuelle ===")
print(tendance)

fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(tendance.index, tendance["score_moyen"],
         marker="o", color="#4C9BE8", label="Score moyen")
ax1.set_ylabel("Score environnemental moyen")
ax1.set_xlabel("Année")

ax2 = ax1.twinx()
ax2.bar(tendance.index, tendance["part_eco"],
        alpha=0.3, color="#4CAF50", label="Part éco (%)")
ax2.set_ylabel("Part éco-responsable (%)")

plt.title("Évolution du score éco et de la part biosourcée")
fig.tight_layout()
plt.savefig("../reports/06_tendance_annuelle.png", dpi=150)
plt.show()